# Inferring
In this lesson, you will infer sentiment and topics from product reviews and news articles.

## Setup

In [ ]:
import anthropic
import os

# os.environ['ANTHROPIC_API_KEY'] = 'YOUR_KEY'
# os.environ['LLM_MODEL'] = 'claude-haiku-4-5'

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())  # read local .env file

# Claude model for the class. Haiku 4.5 is fast and inexpensive for
# high-volume prompt labs; override via LLM_MODEL in .env (e.g. claude-opus-4-8).
MODEL = os.getenv("LLM_MODEL", "claude-haiku-4-5")

In [ ]:
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

def get_completion(prompt, model=MODEL):
    message = client.messages.create(
        model=model,
        max_tokens=2048,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text

## Product review text

In [3]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""

## Sentiment (positive/negative)

In [4]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the product review is positive. The reviewer expresses satisfaction with the lamp's price, the speed of delivery, the ease of assembly, and the customer service provided by the company, Lumina. Despite encountering issues with the product (a broken string and a missing part), the reviewer highlights the company's prompt and helpful response, which contributes to the overall positive sentiment.


In [5]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

Positive


## Identify types of emotions

In [6]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

satisfaction, gratitude, appreciation, relief, contentment


## Identify anger

In [7]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

No


## Extract product and company name from customer reviews

In [8]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

```json
{
  "Item": "lamp",
  "Brand": "Lumina"
}
```


## Doing multiple tasks at once

In [9]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

```json
{
  "Sentiment": "positive",
  "Anger": false,
  "Item": "lamp",
  "Brand": "Lumina"
}
```


## Inferring topics

In [10]:
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""

## Infer 5 topics

In [11]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas.

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

NASA, Employee Satisfaction, Survey Results, Social Security Administration, Government Response


In [12]:
response.split(sep=',')

['NASA',
 ' Employee Satisfaction',
 ' Survey Results',
 ' Social Security Administration',
 ' Government Response']

In [29]:
topic_list = [
    "nasa", "local government", "engineering", 
    "employee satisfaction", "federal government"
]

In [30]:
from difflib import get_close_matches

# Build the dictionary from the list
topic_dict = {i + 1: t for i, t in enumerate(topic_list)}

# First try exact/substring match (case-insensitive)
target = "nasa"
nasa_key = next((k for k, v in topic_dict.items() if target in v.casefold()), None)

# If not found, fall back to fuzzy matching to catch typos like "masa"
if nasa_key is None:
    lower_vals = [v.casefold() for v in topic_list]
    match = get_close_matches(target, lower_vals, n=1, cutoff=0.7)  # 0.7 works for "masa"≈"nasa"
    if match:
        idx = lower_vals.index(match[0])
        nasa_key = idx + 1

if nasa_key is not None:
    print(f"ALERT: New NASA story at key {nasa_key}! (matched '{topic_dict[nasa_key]}')")
else:
    print("No NASA-like topic found.")

ALERT: New NASA story at key 1! (matched 'nasa')


## Try experimenting on your own!